In [1]:
from typing import TypedDict 

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END, MessagesState  
from langgraph.runtime import Runtime 
from langgraph.checkpoint.memory import InMemorySaver # In memory checkpointing for graph -> Short term memory
from langgraph.store.memory import InMemoryStore  # Long term storage for graph -> Long term memory

In [2]:
class AgentState(MessagesState):
    intent: str 
    order_id: str 
    preferred_language: str

In [3]:
class Context(TypedDict):
    user_id: str 
    plan: str 

In [4]:
def parse_request(state: AgentState, config: RunnableConfig) -> dict: # Node
    user_text = str(state["messages"][-1].content).strip() 
    thread_id = config["configurable"]["thread_id"]

    print(f"[Node 1] thread_id = {thread_id}")
    print(f"[Node 1] message_count = {len(state['messages'])}")
    print(f"[Node 1] existing_order = {state.get('order_id', 'N/A')}") 


    if user_text.startswith("My order ID is"):
        order_id = user_text.removeprefix("My order ID is ").strip() 

        return {
            "order_id": order_id,
            "intent": "save_order"
        }

    if user_text == "Where is my order?":
        return {"intent": "ask_order"}

    if user_text.startswith("Remember my language is "):
        return {"intent": "save_language"}

    if user_text == "Which language do I prefer?":
        return {"intent": "ask_language"}

    return {"intent": "unknown"}

In [5]:
def load_user_memory(
    state: AgentState,
    runtime: Runtime[Context]
) -> dict:
    user_id = runtime.context.get("user_id")
    plan = runtime.context.get("plan") 

    store = runtime.store 

    namespace = ("users", user_id)

    if state["intent"] == "save_language":
        message = str(state["messages"][-1].content).strip()
        language = message.removeprefix("Remember my language is ").strip()
        store.put(
            namespace,
            "language",
            {"value": language}
        )

    memory = store.get(namespace, "language")

    language = (
        memory.value["value"]
        if memory is not None else "not set"
    )

    return {
        "preferred_language": language
    }

In [6]:
def respond(
    state: AgentState,
    config: RunnableConfig,
    runtime: Runtime[Context]
) -> dict:
    thread_id = config["configurable"]["thread_id"]
    user_id = runtime.context.get("user_id")
    intent = state["intent"] 

    if intent == "save_order":
        answer = (
            f"Saved order {state['order_id']} for user {user_id} in thread {thread_id}"
        )

    elif intent == "ask_order":
        if state.get("order_id"):
            answer = (
                f"Your order {state['order_id']} is being processed. "
                "It will be shipped soon."
            )
        else:
            answer = (
                "I couldn't find your order. Please provide your order ID."
            )

    elif intent == "save_language":
        answer = (
            f"Saved your preferred language {state['preferred_language']}"
        )
        
    elif intent == "ask_language":
        if state.get("preferred_language"):
            answer = (
                f"Your preferred language is {state['preferred_language']}"
            )
        else:
            answer = (
                "You haven't set your preferred language yet. "
                "Please set your preferred language."
            )

    else:
        answer = "Ask about your order or language preferences."

    return {
        "messages": [AIMessage(content=answer)]
    }

In [7]:
builder = StateGraph(AgentState, context_schema=Context)

builder.add_node("parse_request", parse_request)
builder.add_node("load_user_memory", load_user_memory)
builder.add_node("respond", respond)

builder.add_edge(START, "parse_request")
builder.add_edge("parse_request", "load_user_memory")
builder.add_edge("load_user_memory", "respond")
builder.add_edge("respond", END)

checkpointer = InMemorySaver()
store = InMemoryStore()

graph = builder.compile(
    checkpointer=checkpointer,
    store=store
)

In [8]:
def chat(user_id, thread_id, message, plan="free"):

    print(f"\nUser {user_id} | Thread {thread_id}")
    print(f"\nInput: {message}")

    result = graph.invoke({
        "messages": [HumanMessage(content=message)]
    },
    config={"configurable": {"thread_id": thread_id}, "tags": ["chat", "demo"]},
    context={"user_id": user_id, "plan": plan}
    )

    # graph has been invoked successfully
    print("Answer: ", result["messages"][-1].content)

    print("Total messages: ", len(result["messages"]))

    return result 

In [9]:
chat("user_a", "a1", "My order ID is 1234567890", "premium")


User user_a | Thread a1

Input: My order ID is 1234567890
[Node 1] thread_id = a1
[Node 1] message_count = 1
[Node 1] existing_order = N/A
Answer:  Saved order 1234567890 for user user_a in thread a1
Total messages:  2


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='3293afae-d953-415d-bbe7-329d5e44d72f'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='b87f7a2c-89d5-4071-8100-53e36ca0af15', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'save_order',
 'order_id': '1234567890',
 'preferred_language': 'not set'}

In [10]:
chat("user_a", "a1", "Where is my order?", "premium")


User user_a | Thread a1

Input: Where is my order?
[Node 1] thread_id = a1
[Node 1] message_count = 3
[Node 1] existing_order = 1234567890
Answer:  Your order 1234567890 is being processed. It will be shipped soon.
Total messages:  4


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='3293afae-d953-415d-bbe7-329d5e44d72f'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='b87f7a2c-89d5-4071-8100-53e36ca0af15', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='3a701408-b3de-4abd-b1f7-517e1ae8b44a'),
  AIMessage(content='Your order 1234567890 is being processed. It will be shipped soon.', additional_kwargs={}, response_metadata={}, id='0d2a25af-cab7-4d3d-b50b-9ada23b61d7c', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_order',
 'order_id': '1234567890',
 'preferred_language': 'not set'}

In [11]:
chat("user_a", "a1", "Remember my language is German", "premium")


User user_a | Thread a1

Input: Remember my language is German
[Node 1] thread_id = a1
[Node 1] message_count = 5
[Node 1] existing_order = 1234567890
Answer:  Saved your preferred language German
Total messages:  6


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='3293afae-d953-415d-bbe7-329d5e44d72f'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='b87f7a2c-89d5-4071-8100-53e36ca0af15', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='3a701408-b3de-4abd-b1f7-517e1ae8b44a'),
  AIMessage(content='Your order 1234567890 is being processed. It will be shipped soon.', additional_kwargs={}, response_metadata={}, id='0d2a25af-cab7-4d3d-b50b-9ada23b61d7c', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Remember my language is German', additional_kwargs={}, response_metadata={}, id='9841f26b-7562-47a7-9bce-40ab8db594c1'),
  AIMessage(content='Saved your preferred language German', additional_kwargs={}, response_metadata={}, id='c58d82ef-4a17-48e9-a01a-c2614451ddda', tool_c

In [12]:
chat("user_a", "a2", "Where is my order?", "premium")


User user_a | Thread a2

Input: Where is my order?
[Node 1] thread_id = a2
[Node 1] message_count = 1
[Node 1] existing_order = N/A
Answer:  I couldn't find your order. Please provide your order ID.
Total messages:  2


{'messages': [HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='49a26216-31fb-4f7a-9de5-3a9f6368dd63'),
  AIMessage(content="I couldn't find your order. Please provide your order ID.", additional_kwargs={}, response_metadata={}, id='abc3ac9c-ce12-4038-b33b-7bdf26ccaa43', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_order',
 'preferred_language': 'German'}

In [13]:
chat("user_a", "a2", "Which language do I prefer?", "premium")


User user_a | Thread a2

Input: Which language do I prefer?
[Node 1] thread_id = a2
[Node 1] message_count = 3
[Node 1] existing_order = N/A
Answer:  Your preferred language is German
Total messages:  4


{'messages': [HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='49a26216-31fb-4f7a-9de5-3a9f6368dd63'),
  AIMessage(content="I couldn't find your order. Please provide your order ID.", additional_kwargs={}, response_metadata={}, id='abc3ac9c-ce12-4038-b33b-7bdf26ccaa43', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Which language do I prefer?', additional_kwargs={}, response_metadata={}, id='2f7be881-3db7-4d6f-bc0f-5ab8b2e2d512'),
  AIMessage(content='Your preferred language is German', additional_kwargs={}, response_metadata={}, id='7c00622f-d3c9-440c-b937-9fc9a1498ff4', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_language',
 'preferred_language': 'German'}

In [14]:
chat("user_b", "b1", "Which language do I prefer?", "premium")


User user_b | Thread b1

Input: Which language do I prefer?
[Node 1] thread_id = b1
[Node 1] message_count = 1
[Node 1] existing_order = N/A
Answer:  Your preferred language is not set
Total messages:  2


{'messages': [HumanMessage(content='Which language do I prefer?', additional_kwargs={}, response_metadata={}, id='99c54398-ba14-4af8-8ad6-9e6f8be9882e'),
  AIMessage(content='Your preferred language is not set', additional_kwargs={}, response_metadata={}, id='c97db552-ebb6-41ef-93b2-71b8c7ffdf9d', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_language',
 'preferred_language': 'not set'}

In [15]:
for thread in ["a1", "a2", "b1"]:
    config = {
        "configurable": {"thread_id": thread},
    }

    snapshot = graph.get_state(config=config)
    state = snapshot.values

    print("===Checkpoints===")

    print("checkpoints", snapshot.config["configurable"].get("checkpoint_id"))
    print("Created at", snapshot.created_at)

    

    print(f"\nThread {thread}:")
    print("State: ", state)
    print("Messages: ", len(state["messages"]))
    print("Intent: ", state["intent"])
    print("Order ID: ", state.get("order_id", "N/A"))
    print("Preferred Language: ", state.get("preferred_language", "N/A"))

===Checkpoints===
checkpoints 1f1e7a42-b17c-69ee-800d-322786d112a8
Created at 2026-11-24T01:12:50.882814+00:00

Thread a1:
State:  {'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='3293afae-d953-415d-bbe7-329d5e44d72f'), AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='b87f7a2c-89d5-4071-8100-53e36ca0af15', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='3a701408-b3de-4abd-b1f7-517e1ae8b44a'), AIMessage(content='Your order 1234567890 is being processed. It will be shipped soon.', additional_kwargs={}, response_metadata={}, id='0d2a25af-cab7-4d3d-b50b-9ada23b61d7c', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Remember my language is German', additional_kwargs={}, response_metadata={}, id='9841f26b-7562-47a7-9bce-40ab8db594c1'), AIMessage(content='Saved your 